In [61]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults
import requests
from langchain_classic import hub
from langsmith import Client
from langchain_classic.agents import AgentExecutor, create_react_agent
import re


In [62]:
llm = ChatOpenAI()

In [63]:
webSearchtool = DuckDuckGoSearchResults()


@tool 
def calculateOld(a: int , b: int, op: str) -> int:
    "performs a (operator) b"
    if(op == '+'):
        return a + b;
    if(op == '-'):
        return a - b;
    if(op == '*'):
        return a * b;
    if(op == '/'):
        return a / b;

    return a + b;

@tool
def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression. Input must be digits and + - * / ( ) only, e.g. '241000000 + 128000000'."""
    if not re.fullmatch(r"[\d+\-*/(). ]+", expression):
        return "Error: numbers and + - * / ( ) only. Do not pass words."
    return str(eval(expression))



In [64]:
webSearchtool.invoke("latest news in india")
# Step 2: Pull the ReAct prompt from LangChain Hub
inputPrompt = Client().pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)


In [65]:
agent = create_react_agent(
llm=llm,
prompt = inputPrompt,
tools=[webSearchtool, calculate]
)

In [66]:
agentExecutor = AgentExecutor(
    agent = agent,
    tools = [webSearchtool, calculate],
    verbose= True
)

In [67]:
response = agentExecutor.invoke({"input": "Across all states in india, calculate the population"})




> Entering new AgentExecutor chain...
I should use a search engine to find the population of all states in India and then calculate the total population.
Action: duckduckgo_results_json
Action Input: "population of all states in India"snippet: Under the Indian Constitution and laws, the states of India are self-governing administrative divisions, each having a state government. The legal power to manage affairs in each state is shared or divided between the particular state government on one hand and the national union government on the other. The union territories are directly governed by the union government; no state level ..., title: States and union territories of India - Wikipedia, link: https://en.wikipedia.org/wiki/States_and_union_territories_of_India, snippet: Just 1% of India's population lives in 12 least populous states or union territories, while 10% of the country's population lives in 21 least populous states or union territories. The largest state, Uttar Pradesh, has

In [69]:
response

{'input': 'Across all states in india, calculate the population',
 'output': 'The total population across all states in India is 2,917,900,000.'}

## Version 2 — `create_tool_calling_agent` (native tool calling)

Same `AgentExecutor` as above. Two things change:

- **No ReAct text protocol.** Tools are sent to the model as a JSON schema instead of being interpolated into `{tools}` / `{tool_names}`, so `Action:` / `Action Input:` and the output parser disappear.
- **Multi-argument tools work.** The model returns `{"a": ..., "b": ..., "op": ...}` as structured data, so `calculateOld(a, b, op)` — the 3-arg tool that fails under ReAct — is used here unchanged.

The prompt becomes a `ChatPromptTemplate`, and `agent_scratchpad` is a list of messages rather than a text transcript.

In [70]:
from langchain_classic.agents import create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [71]:
# No {tools} / {tool_names} needed — the model receives the tool schemas directly.
# agent_scratchpad must be a MessagesPlaceholder, not a string slot.
toolCallingPrompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the tools when they help. "
               "Use the calculateOld tool for arithmetic instead of doing it yourself."),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

In [72]:
# calculateOld is the 3-arg tool — works fine here, unlike with create_react_agent
toolCallingAgent = create_tool_calling_agent(
    llm=llm,
    tools=[webSearchtool, calculateOld],
    prompt=toolCallingPrompt,
)

toolCallingExecutor = AgentExecutor(
    agent=toolCallingAgent,
    tools=[webSearchtool, calculateOld],
    verbose=True,
)

In [73]:
# Proof that multi-arg tools work: watch the trace for a={...}, b={...}, op='+'
response2 = toolCallingExecutor.invoke({"input": "What is 241000000 + 128000000?"})
print(response2["output"])



> Entering new AgentExecutor chain...

Invoking: `calculateOld` with `{'a': 241000000, 'b': 128000000, 'op': '+'}`


369000000241,000,000 + 128,000,000 = 369,000,000

> Finished chain.
241,000,000 + 128,000,000 = 369,000,000


In [74]:
# Search tool through the same executor
response3 = toolCallingExecutor.invoke({"input": "Plan a 3-day trip to Goa. Use search for current information."})
print(response3["output"])



> Entering new AgentExecutor chain...

Invoking: `duckduckgo_results_json` with `{'query': 'Things to do in Goa'}`


snippet: March 31, 2026 - Experience a taste of Goa’s Portuguese past with this heritage walk around the charming Old Latin Quarter in Panjim. Panjim, Goa • 2.5 Hours • From Rs. 1400 ... Paint the town red with this fun Goan cuisine & Feni trail around Panjim’s most iconic local establishments. Panjim, Goa • 3 hours • From Rs. 3750 ... Explore the vibrant culture of the city of Margao on this mouth-watering food trail. Its markets, dominant trades and diverse communities will tell you the story of how Margao evolved into the bustling town it is today., title: Things to do in Goa - Make It Happen, link: https://makeithappen.co.in/things-to-do-in-goa/, snippet: June 16, 2026 - When my host family at Cancio’s House invited me on a speedboat ride along North Goa’s riverine backwaters, I had no idea I was going to whizz along such breathtaking scenery – untouched, devoid of